# 🤖 Auto Train AI — Google Colab Pipeline
**End-to-end AutoML pipeline: Upload CSV → Train Models → Get Best Model**

In [ ]:
# ─── Install dependencies ───────────────────────────────────────
!pip install scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

from google.colab import files

from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, confusion_matrix, r2_score,
    mean_squared_error, ConfusionMatrixDisplay
)

print('✅ All imports successful.')

In [ ]:
# ─── STEP 1: Upload & Load Data ─────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    """Load CSV file into DataFrame."""
    df = pd.read_csv(path)
    print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
    return df

print('Upload your CSV file:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = load_data(filename)
df.head()

In [ ]:
# ─── STEP 2: Set Target Column ──────────────────────────────────
print('Available columns:')
for i, col in enumerate(df.columns):
    print(f'  [{i}] {col}')

TARGET = input('\nEnter target column name: ').strip()
print(f'\n✅ Target set to: {TARGET}')

In [ ]:
# ─── STEP 3: Smart Data Analysis ────────────────────────────────
def compute_health_score(df, target):
    score = 100
    missing_ratio = df.isnull().sum().sum() / (df.shape[0] * df.shape[1])
    score -= min(30, int(missing_ratio * 100))
    dup_ratio = df.duplicated().sum() / len(df)
    score -= min(20, int(dup_ratio * 100))
    if df[target].dtype == object or df[target].nunique() < 15:
        vc = df[target].value_counts(normalize=True)
        imb = 1 - vc.min()
        score -= min(20, int(imb * 30))
    return max(0, score)

def analyze_data(df, target):
    print('=' * 60)
    print('DATA ANALYSIS REPORT')
    print('=' * 60)
    print(f'Shape: {df.shape}')
    print(f'\nMissing Values:')
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print(f'\nDuplicate rows: {df.duplicated().sum()}')
    print(f'\nData types:\n{df.dtypes}')
    print(f'\nSummary Statistics:')
    print(df.describe(include="all"))

    hs = compute_health_score(df, target)
    print(f'\n📊 Dataset Health Score: {hs}/100')

    # Correlation heatmap
    num_df = df.select_dtypes(include=np.number)
    if len(num_df.columns) > 1:
        plt.figure(figsize=(10, 6))
        sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
        plt.title('Correlation Heatmap')
        plt.tight_layout()
        plt.show()

    # Target distribution
    plt.figure(figsize=(7, 3))
    df[target].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title(f'Target Distribution: {target}')
    plt.tight_layout()
    plt.show()

analyze_data(df, TARGET)

In [ ]:
# ─── STEP 4: Detect Problem Type ────────────────────────────────
def detect_problem_type(df, target):
    col = df[target]
    if col.dtype == object or col.nunique() < 15:
        pt = 'classification'
        reason = f'Target has dtype={col.dtype}, {col.nunique()} unique values (< 15 threshold)'
    else:
        pt = 'regression'
        reason = f'Target has dtype={col.dtype}, {col.nunique()} unique values (>= 15 threshold)'
    return pt, reason

PROBLEM_TYPE, reason = detect_problem_type(df, TARGET)
print(f'\n🎯 Problem Type: {PROBLEM_TYPE.upper()}')
print(f'   Reason: {reason}')

In [ ]:
# ─── STEP 5: Intelligent Preprocessing ──────────────────────────
def preprocess_data(df, target):
    df = df.copy()
    before = len(df)
    df.drop_duplicates(inplace=True)
    print(f'Removed {before - len(df)} duplicate rows.')

    y = df[target].copy()
    X = df.drop(columns=[target])

    if y.dtype == object:
        le = LabelEncoder()
        y = pd.Series(le.fit_transform(y), name=target)
        print('Target label-encoded.')

    to_drop = []
    for col in X.columns:
        if X[col].dtype == object and X[col].nunique() / len(X) > 0.5:
            to_drop.append(col)
            print(f'Dropping high-cardinality: {col}')
        if X[col].nunique() <= 1:
            to_drop.append(col)
            print(f'Dropping constant: {col}')
    X.drop(columns=list(set(to_drop)), inplace=True)

    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')

    transformers = []
    if num_cols:
        transformers.append(('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols))
    if cat_cols:
        transformers.append(('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols))

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train_p = preprocessor.fit_transform(X_train)
    X_test_p = preprocessor.transform(X_test)

    feature_names = num_cols.copy()
    if cat_cols:
        ohe = preprocessor.named_transformers_['cat']['enc']
        feature_names += ohe.get_feature_names_out(cat_cols).tolist()

    print(f'\n✅ Preprocessing done. Train: {X_train_p.shape}, Test: {X_test_p.shape}')
    return X_train_p, X_test_p, y_train, y_test, preprocessor, feature_names

X_train, X_test, y_train, y_test, preprocessor, feature_names = preprocess_data(df, TARGET)

In [ ]:
# ─── STEP 6: Train Models ────────────────────────────────────────
def get_models(problem_type):
    if problem_type == 'classification':
        return {
            'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
            'Decision Tree': DecisionTreeClassifier(random_state=42),
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'K-Nearest Neighbors': KNeighborsClassifier(),
        }
    return {
        'Linear Regression': LinearRegression(),
        'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
        'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    }

def train_models(X_train, X_test, y_train, y_test, problem_type):
    models = get_models(problem_type)
    results, trained = [], {}
    metric = 'Accuracy' if problem_type == 'classification' else 'R² Score'

    for name, model in models.items():
        t0 = time.time()
        model.fit(X_train, y_train)
        elapsed = time.time() - t0
        y_pred = model.predict(X_test)
        score = accuracy_score(y_test, y_pred) if problem_type == 'classification' \
                else r2_score(y_test, y_pred)
        row = {'Model': name, metric: round(score, 4), 'Train Time (s)': round(elapsed, 3)}
        if problem_type == 'regression':
            row['RMSE'] = round(np.sqrt(mean_squared_error(y_test, y_pred)), 4)
        results.append(row)
        trained[name] = model
        print(f'  {name}: {metric}={score:.4f} | {elapsed:.3f}s')

    df_r = pd.DataFrame(results).sort_values(metric, ascending=False).reset_index(drop=True)
    return df_r, trained

print('Training models...\n')
df_results, trained = train_models(X_train, X_test, y_train, y_test, PROBLEM_TYPE)
print('\n🏆 Leaderboard:')
print(df_results.to_string(index=False))

In [ ]:
# ─── STEP 7: Cross Validation ───────────────────────────────────
def cross_validate_models(X_train, y_train, problem_type):
    models = get_models(problem_type)
    scoring = 'accuracy' if problem_type == 'classification' else 'r2'
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rows = []
    for name, model in models.items():
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scoring)
        rows.append({'Model': name, 'CV Mean': round(scores.mean(),4), 'CV Std': round(scores.std(),4)})
    return pd.DataFrame(rows).sort_values('CV Mean', ascending=False).reset_index(drop=True)

print('Running 5-fold cross-validation...')
df_cv = cross_validate_models(X_train, y_train, PROBLEM_TYPE)
print(df_cv.to_string(index=False))

In [ ]:
# ─── STEP 8: Hyperparameter Tuning ──────────────────────────────
PARAM_GRIDS = {
    'Random Forest': {'n_estimators':[50,100,200],'max_depth':[None,5,10],'min_samples_split':[2,5]},
    'Random Forest Regressor': {'n_estimators':[50,100,200],'max_depth':[None,5,10],'min_samples_split':[2,5]},
    'Decision Tree': {'max_depth':[None,3,5,10],'min_samples_split':[2,5,10]},
    'Decision Tree Regressor': {'max_depth':[None,3,5,10],'min_samples_split':[2,5,10]},
    'Logistic Regression': {'C':[0.01,0.1,1,10],'solver':['lbfgs','liblinear']},
    'K-Nearest Neighbors': {'n_neighbors':[3,5,7,11],'weights':['uniform','distance']},
}

scoring = 'accuracy' if PROBLEM_TYPE == 'classification' else 'r2'
top2 = df_results['Model'].head(2).tolist()

for name in top2:
    if name not in PARAM_GRIDS or not PARAM_GRIDS[name]:
        print(f'{name}: no tuning needed.')
        continue
    print(f'Tuning {name}...')
    gs = RandomizedSearchCV(trained[name].__class__(random_state=42) if hasattr(trained[name],'random_state') else trained[name].__class__(),
                            PARAM_GRIDS[name], n_iter=10, cv=3, scoring=scoring, random_state=42, n_jobs=-1)
    gs.fit(X_train, y_train)
    print(f'  Best score: {gs.best_score_:.4f} | Params: {gs.best_params_}')
    trained[name] = gs.best_estimator_

In [ ]:
# ─── STEP 9: Select Best Model ───────────────────────────────────
metric_col = 'Accuracy' if PROBLEM_TYPE == 'classification' else 'R² Score'
best_name = df_results.iloc[0]['Model']
best_score = df_results.iloc[0][metric_col]
best_model = trained[best_name]

print(f'\n🏆 BEST MODEL: {best_name}')
print(f'   {metric_col}: {best_score:.4f}')
print(f'   Reason: Highest {metric_col} on held-out test set, consistent CV performance.')

In [ ]:
# ─── STEP 10: Feature Importance ────────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    fi = pd.DataFrame({'Feature': feature_names, 'Importance': best_model.feature_importances_})\
           .sort_values('Importance', ascending=False).head(15)
    plt.figure(figsize=(8, 4))
    plt.barh(fi['Feature'], fi['Importance'], color='steelblue')
    plt.xlabel('Importance')
    plt.title(f'Feature Importance — {best_name}')
    plt.tight_layout()
    plt.show()
    print(f'\n💡 Top feature: "{fi.iloc[0]["Feature"]}" contributes most to the prediction.')
else:
    print('Feature importance not available for this model.')

In [ ]:
# ─── STEP 11: Model Comparison Chart ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['gold'] + ['steelblue'] * (len(df_results) - 1)
axes[0].bar(df_results['Model'], df_results[metric_col], color=colors, edgecolor='white')
axes[0].set_title(f'Model Scores — {metric_col}')
axes[0].set_ylabel(metric_col)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

axes[1].bar(df_results['Model'], df_results['Train Time (s)'], color='coral', edgecolor='white')
axes[1].set_title('Training Time (seconds)')
axes[1].set_ylabel('Seconds')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# ─── STEP 12: Export Model ───────────────────────────────────────
model_filename = f'best_model_{best_name.replace(" ","_")}.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(best_model, f)

with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

print(f'✅ Saved {model_filename} and preprocessor.pkl')
files.download(model_filename)
files.download('preprocessor.pkl')

In [ ]:
# ─── STEP 13: Make a Prediction ─────────────────────────────────
# Example: build an input row manually using the original feature columns.
# Replace with actual values from your dataset.

# Get feature columns (exclude target)
feature_cols = [c for c in df.columns if c != TARGET]
sample = df[feature_cols].iloc[0:1].copy()  # Use first row as example input

print('Sample input:')
print(sample)

processed_input = preprocessor.transform(sample)
prediction = best_model.predict(processed_input)[0]
print(f'\n🎯 Prediction: {prediction}')

if PROBLEM_TYPE == 'classification' and hasattr(best_model, 'predict_proba'):
    proba = best_model.predict_proba(processed_input)[0]
    print('\nClass Probabilities:')
    for cls, p in zip(best_model.classes_, proba):
        print(f'  Class {cls}: {p*100:.1f}%')